# UPI Product Analytics Data Enrichment

## 1. Import Libraries

In [4]:
import pandas as pd
import numpy as np 

## 2. Load Raw Dataset

In [7]:
df = pd.read_csv("../data/raw/upi_transactions_2024.csv")

In [8]:
df.head()

,transaction id,timestamp,transaction type,merchant_category,amount (INR),transaction_status,sender_age_group,receiver_age_group,sender_state,sender_bank,receiver_bank,device_type,network_type,fraud_flag,hour_of_day,day_of_week,is_weekend
0,TXN0000000001,2024-10-08 15:17:28,P2P,Entertainment,868,SUCCESS,26-35,18-25,Delhi,Axis,SBI,Android,4G,0,15,Tuesday,0
1,TXN0000000002,2024-04-11 06:56:00,P2M,Grocery,1011,SUCCESS,26-35,26-35,Uttar Pradesh,ICICI,Axis,iOS,4G,0,6,Thursday,0
2,TXN0000000003,2024-04-02 13:27:18,P2P,Grocery,477,SUCCESS,26-35,36-45,Karnataka,Yes Bank,PNB,Android,4G,0,13,Tuesday,0
3,TXN0000000004,2024-01-07 10:09:17,P2P,Fuel,2784,SUCCESS,26-35,26-35,Delhi,ICICI,PNB,Android,5G,0,10,Sunday,1
4,TXN0000000005,2024-01-23 19:04:23,P2P,Shopping,990,SUCCESS,26-35,18-25,Delhi,Axis,Yes Bank,iOS,WiFi,0,19,Tuesday,0


In [9]:
df.columns

Index(['transaction id', 'timestamp', 'transaction type', 'merchant_category',
       'amount (INR)', 'transaction_status', 'sender_age_group',
       'receiver_age_group', 'sender_state', 'sender_bank', 'receiver_bank',
       'device_type', 'network_type', 'fraud_flag', 'hour_of_day',
       'day_of_week', 'is_weekend'],
      dtype='str')

In [10]:
df.shape

(250000, 17)

## 3. Engineer User ID

In [11]:
num_users = 25000

In [12]:
user_ids = [f"U{i:05d}" for i in range(1, num_users + 1)]

In [13]:
user_ids[:10]

['U00001',
 'U00002',
 'U00003',
 'U00004',
 'U00005',
 'U00006',
 'U00007',
 'U00008',
 'U00009',
 'U00010']

In [14]:
len(user_ids)

25000

In [15]:
df["user_id"]=np.random.choice(user_ids,size=len(df))

In [16]:
df.head(2)

,transaction id,timestamp,transaction type,merchant_category,amount (INR),transaction_status,sender_age_group,receiver_age_group,sender_state,sender_bank,receiver_bank,device_type,network_type,fraud_flag,hour_of_day,day_of_week,is_weekend,user_id
0,TXN0000000001,2024-10-08 15:17:28,P2P,Entertainment,868,SUCCESS,26-35,18-25,Delhi,Axis,SBI,Android,4G,0,15,Tuesday,0,U23291
1,TXN0000000002,2024-04-11 06:56:00,P2M,Grocery,1011,SUCCESS,26-35,26-35,Uttar Pradesh,ICICI,Axis,iOS,4G,0,6,Thursday,0,U13274


In [17]:
user_data = df[df['user_id'] == "U24990"]


In [18]:
user_count = df['user_id'].value_counts().head(5)

In [19]:
df = df[["user_id"] + [col for col in df.columns if col != "user_id"]]

In [ ]:


users_master = pd.DataFrame({
    "user_id": user_ids,
})

## 4. Engineer App Name

In [21]:
app = [
    "PhonePe",
    "Google Pay",
    "Paytm",
    "BHIM"
]

In [22]:
app_probabilities = [
    0.40,
    0.35,
    0.15,
    0.10
]

In [23]:
users_master["app_name"] = np.random.choice(app,size=len(users_master),p=app_probabilities)

In [24]:
users_master.head()

,user_id,app_name
0,U00001,Google Pay
1,U00002,Paytm
2,U00003,PhonePe
3,U00004,Google Pay
4,U00005,PhonePe


In [25]:
users_master["app_name"].value_counts()

app_name
PhonePe       9994
Google Pay    8697
Paytm         3815
BHIM          2494
Name: count, dtype: int64

In [26]:
df = df.merge(
    users_master,
    on="user_id",
    how="left"
)

In [27]:
df.head(2)

,user_id,transaction id,timestamp,transaction type,merchant_category,amount (INR),transaction_status,sender_age_group,receiver_age_group,sender_state,sender_bank,receiver_bank,device_type,network_type,fraud_flag,hour_of_day,day_of_week,is_weekend,app_name
0,U23291,TXN0000000001,2024-10-08 15:17:28,P2P,Entertainment,868,SUCCESS,26-35,18-25,Delhi,Axis,SBI,Android,4G,0,15,Tuesday,0,Google Pay
1,U13274,TXN0000000002,2024-04-11 06:56:00,P2M,Grocery,1011,SUCCESS,26-35,26-35,Uttar Pradesh,ICICI,Axis,iOS,4G,0,6,Thursday,0,Paytm


In [28]:
df[df['user_id'] == "U21151"].head(2)

,user_id,transaction id,timestamp,transaction type,merchant_category,amount (INR),transaction_status,sender_age_group,receiver_age_group,sender_state,sender_bank,receiver_bank,device_type,network_type,fraud_flag,hour_of_day,day_of_week,is_weekend,app_name
20891,U21151,TXN0000020892,2024-01-30 13:32:10,P2P,Grocery,167,SUCCESS,26-35,26-35,Maharashtra,SBI,ICICI,Android,4G,0,13,Tuesday,0,PhonePe
80522,U21151,TXN0000080523,2024-09-29 20:07:21,P2P,Fuel,780,SUCCESS,18-25,36-45,Karnataka,SBI,ICICI,iOS,3G,0,20,Sunday,1,PhonePe


## 5. Engineer User Signup Date

In [29]:
df['timestamp'] = pd.to_datetime(
    df['timestamp']
)

In [30]:
df.head(2)

,user_id,transaction id,timestamp,transaction type,merchant_category,amount (INR),transaction_status,sender_age_group,receiver_age_group,sender_state,sender_bank,receiver_bank,device_type,network_type,fraud_flag,hour_of_day,day_of_week,is_weekend,app_name
0,U23291,TXN0000000001,2024-10-08 15:17:28,P2P,Entertainment,868,SUCCESS,26-35,18-25,Delhi,Axis,SBI,Android,4G,0,15,Tuesday,0,Google Pay
1,U13274,TXN0000000002,2024-04-11 06:56:00,P2M,Grocery,1011,SUCCESS,26-35,26-35,Uttar Pradesh,ICICI,Axis,iOS,4G,0,6,Thursday,0,Paytm


In [31]:
df['timestamp'].dtype

dtype('<M8[us]')

In [32]:
first_txn = df.groupby('user_id')['timestamp'].min()

In [33]:
first_txn = first_txn.reset_index()

In [34]:
first_txn.rename( columns={ "timestamp":"first_transaction_date" }, inplace=True )

In [35]:
first_txn.head(2)

,user_id,first_transaction_date
0,U00001,2024-03-01 02:39:56
1,U00002,2024-01-30 18:10:12


In [36]:
users_master = users_master.merge( first_txn, on="user_id", how="left" )

In [37]:
signup_offset = np.random.randint( 1, 180, size=len(users_master) )

In [38]:
users_master["user_signup_date"] = ( users_master["first_transaction_date"] - pd.to_timedelta( signup_offset, unit="D" ) )

In [39]:
users_master.head()

,user_id,app_name,first_transaction_date,user_signup_date
0,U00001,Google Pay,2024-03-01 02:39:56,2024-02-17 02:39:56
1,U00002,Paytm,2024-01-30 18:10:12,2024-01-06 18:10:12
2,U00003,PhonePe,2024-01-11 09:01:32,2023-07-27 09:01:32
3,U00004,Google Pay,2024-02-06 12:14:31,2023-12-03 12:14:31
4,U00005,PhonePe,2024-02-26 14:35:50,2023-11-24 14:35:50


In [40]:
df = df.merge( users_master[ [ "user_id", "user_signup_date" ] ], on="user_id", how="left" )

In [41]:
df.head(2)

,user_id,transaction id,timestamp,transaction type,merchant_category,amount (INR),transaction_status,sender_age_group,receiver_age_group,sender_state,sender_bank,receiver_bank,device_type,network_type,fraud_flag,hour_of_day,day_of_week,is_weekend,app_name,user_signup_date
0,U23291,TXN0000000001,2024-10-08 15:17:28,P2P,Entertainment,868,SUCCESS,26-35,18-25,Delhi,Axis,SBI,Android,4G,0,15,Tuesday,0,Google Pay,2024-01-17 21:39:40
1,U13274,TXN0000000002,2024-04-11 06:56:00,P2M,Grocery,1011,SUCCESS,26-35,26-35,Uttar Pradesh,ICICI,Axis,iOS,4G,0,6,Thursday,0,Paytm,2024-01-11 11:51:54


In [42]:
( df["timestamp"] >= df["user_signup_date"] ).all()

np.True_

## 6. Engineer KYC Status

In [43]:
kyc_value = [
    "Verified",
    "Pending",
    "Not Verified"
]

In [44]:
kyc_probabilities = [
    0.70,
    0.20,
    0.10
]

In [45]:
users_master["kyc_status"] = np.random.choice(
    kyc_value,
    size = len(users_master),
    p = kyc_probabilities
)

In [46]:
users_master['kyc_status'].value_counts()

kyc_status
Verified        17409
Pending          5065
Not Verified     2526
Name: count, dtype: int64

## 7. Engineer Device Type

In [50]:
devices = [
    "Android",
    "iOS"
]

In [51]:
device_probabilities = [
    0.92,
    0.08
]

In [52]:
users_master['device_type'] = np.random.choice(
    devices,
    size = len(users_master),
    p = device_probabilities
)

In [53]:
users_master['device_type'].value_counts()

device_type
Android    22986
iOS         2014
Name: count, dtype: int64

In [54]:
df = df.merge(
    users_master[ [ "user_id", "kyc_status", "device_type" ] ],
    on="user_id",
    how="left"
)

In [55]:
df = df.reset_index(drop=True)

In [57]:
df[df["user_id"]=="U00001"].head(2)

,user_id,transaction id,timestamp,transaction type,merchant_category,amount (INR),transaction_status,sender_age_group,receiver_age_group,sender_state,...,device_type_x,network_type,fraud_flag,hour_of_day,day_of_week,is_weekend,app_name,user_signup_date,kyc_status,device_type_y
7917,U00001,TXN0000007918,2024-03-13 14:18:45,P2P,Food,298,SUCCESS,18-25,18-25,Karnataka,...,iOS,4G,0,14,Wednesday,0,Google Pay,2024-02-17 02:39:56,Verified,Android
8472,U00001,TXN0000008473,2024-06-25 10:04:33,Recharge,Grocery,1548,SUCCESS,26-35,26-35,Tamil Nadu,...,Web,4G,0,10,Tuesday,0,Google Pay,2024-02-17 02:39:56,Verified,Android


## 8. Engineer Merchant Category

In [58]:
merchant_categories = [ "P2P", "Utilities", "Recharge", "Grocery", "Food", "E-commerce", "Fuel", "Healthcare", "Travel", "Entertainment" ]

In [59]:
merchant_probabilities = [ 0.25, 0.15, 0.10, 0.10, 0.10, 0.10, 0.07, 0.05, 0.05, 0.03 ]

In [60]:
df["merchant_category"] = np.random.choice( merchant_categories, size=len(df), p=merchant_probabilities )

In [62]:
df.head(2)

,user_id,transaction id,timestamp,transaction type,merchant_category,amount (INR),transaction_status,sender_age_group,receiver_age_group,sender_state,...,device_type_x,network_type,fraud_flag,hour_of_day,day_of_week,is_weekend,app_name,user_signup_date,kyc_status,device_type_y
0,U23291,TXN0000000001,2024-10-08 15:17:28,P2P,P2P,868,SUCCESS,26-35,18-25,Delhi,...,Android,4G,0,15,Tuesday,0,Google Pay,2024-01-17 21:39:40,Verified,Android
1,U13274,TXN0000000002,2024-04-11 06:56:00,P2M,Grocery,1011,SUCCESS,26-35,26-35,Uttar Pradesh,...,iOS,4G,0,6,Thursday,0,Paytm,2024-01-11 11:51:54,Verified,iOS


In [63]:
df["merchant_category"].value_counts()

merchant_category
P2P              62465
Utilities        37494
Food             25194
Grocery          25098
Recharge         25015
E-commerce       24979
Fuel             17365
Travel           12435
Healthcare       12363
Entertainment     7592
Name: count, dtype: int64

In [ ]:
df[df["user_id"]=="U00001"].head(2)

,user_id,transaction id,timestamp,transaction type,merchant_category,amount (INR),transaction_status,sender_age_group,receiver_age_group,sender_state,...,device_type_x,network_type,fraud_flag,hour_of_day,day_of_week,is_weekend,app_name,user_signup_date,kyc_status,device_type_y
7917,U00001,TXN0000007918,2024-03-13 14:18:45,P2P,E-commerce,298,SUCCESS,18-25,18-25,Karnataka,...,iOS,4G,0,14,Wednesday,0,Google Pay,2024-02-17 02:39:56,Verified,Android
8472,U00001,TXN0000008473,2024-06-25 10:04:33,Recharge,Fuel,1548,SUCCESS,26-35,26-35,Tamil Nadu,...,Web,4G,0,10,Tuesday,0,Google Pay,2024-02-17 02:39:56,Verified,Android
55834,U00001,TXN0000055835,2024-03-10 13:30:33,Recharge,Recharge,429,SUCCESS,18-25,26-35,Tamil Nadu,...,Android,4G,0,13,Sunday,1,Google Pay,2024-02-17 02:39:56,Verified,Android
81748,U00001,TXN0000081749,2024-08-26 07:28:28,P2M,E-commerce,1715,SUCCESS,26-35,46-55,Uttar Pradesh,...,iOS,4G,0,7,Monday,0,Google Pay,2024-02-17 02:39:56,Verified,Android
117955,U00001,TXN0000117956,2024-10-25 18:15:27,P2P,Food,194,SUCCESS,46-55,18-25,Rajasthan,...,Android,WiFi,0,18,Friday,0,Google Pay,2024-02-17 02:39:56,Verified,Android
121505,U00001,TXN0000121506,2024-12-21 18:23:54,Recharge,E-commerce,185,SUCCESS,18-25,26-35,Telangana,...,Android,5G,0,18,Saturday,1,Google Pay,2024-02-17 02:39:56,Verified,Android
127010,U00001,TXN0000127011,2024-06-20 17:08:11,Bill Payment,P2P,306,SUCCESS,36-45,36-45,Telangana,...,Android,3G,0,17,Thursday,0,Google Pay,2024-02-17 02:39:56,Verified,Android
156968,U00001,TXN0000156969,2024-05-09 07:29:10,P2P,Utilities,2281,SUCCESS,46-55,18-25,Tamil Nadu,...,iOS,5G,0,7,Thursday,0,Google Pay,2024-02-17 02:39:56,Verified,Android
162540,U00001,TXN0000162541,2024-08-02 09:17:53,P2P,Utilities,4608,SUCCESS,26-35,36-45,Tamil Nadu,...,iOS,4G,0,9,Friday,0,Google Pay,2024-02-17 02:39:56,Verified,Android
181600,U00001,TXN0000181601,2024-03-01 02:39:56,P2M,Utilities,1140,SUCCESS,36-45,36-45,Telangana,...,Android,5G,0,2,Friday,0,Google Pay,2024-02-17 02:39:56,Verified,Android


In [67]:
df = df.reset_index(drop=True)

## 9. Save Final Enriched Dataset

In [69]:
df.to_csv(
    "../data/enriched/upi_transactions_enriched.csv",
    index=False
)